# Traduction Éwé → Français (NLLB-200 + LoRA)

Ce notebook est la **version française** de `traduction.ipynb`.
La seule différence de fond est la **langue cible** :

```python
SRC_LANG = "ewe_Latn"   # Éwé (source)
TGT_LANG = "fra_Latn"   # Français (cible)
```

Le jeu de données `romaricnadjire/ewe-nllb-translation` contient déjà des paires
`ewe_Latn ↔ fra_Latn` (≈ 23 600 exemples) ; on réutilise donc exactement le même
pipeline (nettoyage, baseline, LoRA, entraînement avec reprise sur checkpoint,
évaluation BLEU / chrF++). Tous les réglages mémoire pour Kaggle T4 sont conservés.


## 1. Chargement du dataset

Format de chaque ligne JSONL :
```json
{"translation": {"ewe_Latn": "...", "eng_Latn": "..."}}
```
Certaines lignes ont `fra_Latn` au lieu de `eng_Latn` — on les filtre.

In [ ]:
!pip install -q evaluate sacrebleu

In [ ]:
import json
import os
import re
from pathlib import Path

# Forcer l'utilisation d'UN SEUL GPU : sur Kaggle T4 x2, le Trainer active
# automatiquement DataParallel qui réplique le modèle et sature la VRAM.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Réduit la fragmentation mémoire (recommandé par le message OOM de PyTorch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass  # local : huggingface-cli login ou variable d'environnement HF_TOKEN

ds_raw = load_dataset(
    "romaricnadjire/ewe-nllb-translation",
    data_files={
        "train":      "train.jsonl",
        "validation": "validation.jsonl",
        "test":       "test.jsonl",
    },
    token=True,
)

ds_raw

## 2. Configuration

Tous les hyperparamètres sont centralisés ici pour faciliter les expériences.

In [ ]:
# ── Modèle & chemins ────────────────────────────────────────────────────────
MODEL_NAME   = "facebook/nllb-200-distilled-600M"
OUTPUT_DIR   = "./output/nllb-ewe-fra"
ADAPTER_DIR  = "./output/nllb-ewe-fra/adapter"
RESULTS_FILE = "./output/resultats_evaluation_fra.json"

# ── Paires de langues (codes BCP-47 NLLB) ────────────────────────────────────
SRC_LANG = "ewe_Latn"   # Éwé
TGT_LANG = "fra_Latn"   # Français (langue cible)

# ── Données ──────────────────────────────────────────────────────────────────
DATA_DIR = Path("data/processed/translation")

# ── Tokenizer ────────────────────────────────────────────────────────────────
# max_length : longueur en TOKENS. Trop grand → explosion VRAM (attention O(n²)).
MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 128

# ── Entraînement ─────────────────────────────────────────────────────────────
# learning_rate : taille du pas de l'optimiseur.
#   Trop grand → la loss diverge ; trop petit → convergence lente.
#   Plus élevé qu'habituellement (3e-4 vs 5e-5) car seul LoRA est entraîné.
LEARNING_RATE = 3e-4

# per_device_train_batch_size : exemples par GPU par micro-étape.
#   Réduit à 4 pour tenir dans les 14.5 Go d'un T4 (était 8 → OOM).
#   Le batch effectif est maintenu via GRAD_ACCUM_STEPS.
BATCH_SIZE_TRAIN = 4
BATCH_SIZE_EVAL  = 8

# gradient_accumulation_steps : accumule N micro-gradients avant un update.
#   Batch effectif = BATCH_SIZE_TRAIN × GRAD_ACCUM_STEPS = 4 × 4 = 16.
GRAD_ACCUM_STEPS = 4

# num_train_epochs : tours complets sur le dataset d'entraînement.
NUM_EPOCHS = 3

# warmup_ratio : fraction des steps dédiée à la montée progressive du LR.
#   Évite un démarrage brutal qui endommagerait les poids pré-entraînés.
WARMUP_RATIO = 0.06

# weight_decay : pénalité L2 légère sur les poids — réduit l'overfitting.
WEIGHT_DECAY = 0.01

# ── LoRA ──────────────────────────────────────────────────────────────────────
# r (rang) : dimension des matrices d'adaptation A et B (ΔW = BA, rang r).
#   Plus r est grand → plus de capacité, mais plus coûteux. Plage : 4–64.
LORA_R = 16

# lora_alpha : facteur d'échelle appliqué à ΔW (mise à l'échelle par α/r ou α/√r).
#   Convention courante : alpha = 2 × r.
LORA_ALPHA = 32

# lora_dropout : régularisation sur les couches LoRA — utile sur petits corpus.
LORA_DROPOUT = 0.05

# target_modules : quelles projections linéaires adapter.
#   Pour NLLB (architecture M2M-100) : q_proj et v_proj sont les plus impactants.
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# ── Évaluation rapide ─────────────────────────────────────────────────────────
BASELINE_SAMPLE = 200   # exemples pour l'éval baseline (validation, rapide)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration OK")
print(f"  Paire          : {SRC_LANG} → {TGT_LANG}")
print(f"  Batch effectif : {BATCH_SIZE_TRAIN} × {GRAD_ACCUM_STEPS} = {BATCH_SIZE_TRAIN * GRAD_ACCUM_STEPS}")

## 3. Nettoyage du bruit

On retire ici deux bruits simples mais nuisibles :
- les exemples où la source et la cible sont identiques (copie au lieu de traduction)
- les cibles anglaises qui contiennent des caractères typiques de l'éwé

In [ ]:
# Caractères typiques de l'éwé — ne doivent pas apparaître dans une cible non-éwé.
EWE_CHARS = set("ŋɖɔɛʋƒãẽĩõũ")

# Référence biblique seule : "15:1-33", "24:1-33", etc.
BIBLE_REF_RE = re.compile(r"^\s*\d{1,3}:\d{1,3}(?:-\d{1,3})?\s*$")

def is_noisy(example, src_lang=SRC_LANG, tgt_lang=TGT_LANG):
    t   = example["translation"]
    src = (t.get(src_lang) or "").strip()
    tgt = (t.get(tgt_lang) or "").strip()

    # 1. Source identique à la cible (copie sans traduction)
    if src and tgt and src == tgt:
        return True

    # 2. Référence biblique seule des deux côtés
    if BIBLE_REF_RE.match(src) and src == tgt:
        return True

    # 3. Caractères éwé dans une cible non-éwé
    if tgt_lang != "ewe_Latn" and sum(ch in EWE_CHARS for ch in tgt) >= 2:
        return True

    return False

raw_before_cleaning = ds_raw
ds_raw = ds_raw.filter(lambda ex: not is_noisy(ex, SRC_LANG, TGT_LANG))

print("Nettoyage terminé.")
for split in ["train", "validation", "test"]:
    removed = len(raw_before_cleaning[split]) - len(ds_raw[split])
    print(f"  {split:<10} : supprimés={removed} | restants={len(ds_raw[split])}")

## 4. Évaluation Baseline

On mesure les performances du modèle **sans aucun fine-tuning** sur notre dataset.  
Ce score de référence permettra de quantifier l'apport de l'entraînement.

**Métriques** :
- **BLEU** : recouvrement de n-grammes de mots (0–100, ↑ = mieux)
- **chrF++** : n-grammes de caractères + bigrammes de mots — plus robuste pour les langues à morphologie riche comme l'éwé

In [ ]:
sacrebleu_metric = evaluate.load("sacrebleu")
chrf_metric      = evaluate.load("chrf")

print("Chargement du modèle baseline…")
tokenizer_base = AutoTokenizer.from_pretrained(MODEL_NAME)
model_base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model_base.eval()
print("Modèle chargé.")

In [ ]:
def translate_batch(model, tokenizer, sources, src_lang, tgt_lang,
                    batch_size=16, max_new_tokens=128):
    """Génère les traductions pour une liste de phrases source."""
    tokenizer.src_lang = src_lang
    forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)
    all_preds = []

    for i in range(0, len(sources), batch_size):
        batch = sources[i : i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LEN,
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos,
                max_new_tokens=max_new_tokens,
                num_beams=4,
            )
        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        all_preds.extend(decoded)

    return all_preds


def compute_metrics_on_split(model, tokenizer, dataset_split, n_samples=None):
    """Calcule BLEU et chrF++ sur un split (ou n_samples exemples)."""
    if n_samples:
        dataset_split = dataset_split.select(range(min(n_samples, len(dataset_split))))

    # Filtrer les paires où source ou référence est None/vide
    pairs = [
        (ex["translation"].get(SRC_LANG), ex["translation"].get(TGT_LANG))
        for ex in dataset_split
        if ex["translation"].get(SRC_LANG) and ex["translation"].get(TGT_LANG)
    ]
    sources    = [p[0] for p in pairs]
    references = [p[1] for p in pairs]

    print(f"  Génération de {len(sources)} traductions…")
    predictions = translate_batch(model, tokenizer, sources, SRC_LANG, TGT_LANG)

    bleu = sacrebleu_metric.compute(
        predictions=predictions, references=[[r] for r in references]
    )
    chrf = chrf_metric.compute(
        predictions=predictions, references=[[r] for r in references], word_order=2
    )

    return {
        "bleu":   round(bleu["score"], 2),
        "chrf++": round(chrf["score"], 2),
        "n":      len(sources),
    }


In [ ]:
print("=== BASELINE — validation rapide ===")
baseline_val = compute_metrics_on_split(
    model_base, tokenizer_base, ds_raw["validation"], n_samples=BASELINE_SAMPLE
)
print(f"  BLEU   : {baseline_val['bleu']}")
print(f"  chrF++ : {baseline_val['chrf++']}")

In [ ]:
print("=== BASELINE — test complet ===")
baseline_test = compute_metrics_on_split(model_base, tokenizer_base, ds_raw["test"])
print(f"  BLEU   : {baseline_test['bleu']}")
print(f"  chrF++ : {baseline_test['chrf++']}")

# ── Sauvegarde des résultats baseline ─────────────────────────────────────────
results = {
    "modele": MODEL_NAME,
    "paire":  f"{SRC_LANG} → {TGT_LANG}",
    "baseline": {
        "validation_sample": baseline_val,
        "test":              baseline_test,
    },
    "fine_tune": {},
}

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nBaseline sauvegardé dans {RESULTS_FILE}")

# Libérer la VRAM avant l'entraînement
del model_base, tokenizer_base
if device == "cuda":
    torch.cuda.empty_cache()

## 5. Prétraitement / Tokenisation

Le tokenizer transforme chaque phrase en `input_ids` (liste d'entiers).  
- `max_length` tronque les phrases trop longues  
- Le padding est géré dynamiquement par `DataCollatorForSeq2Seq` (plus efficace que de pré-padder)  
- Les tokens de padding dans les **labels** sont remplacés par **-100** → ignorés par la cross-entropy

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.src_lang = SRC_LANG
FORCED_BOS_TOKEN_ID = tokenizer.convert_tokens_to_ids(TGT_LANG)


# Filtrer les exemples qui n'ont pas les deux langues
# (certaines lignes ont fra_Latn au lieu de eng_Latn → KeyError sinon)
ds_raw = ds_raw.filter(
    lambda ex: ex["translation"].get(SRC_LANG) and ex["translation"].get(TGT_LANG)
)
print("Après filtrage des paires incomplètes :")
for split in ["train", "validation", "test"]:
    print(f"  {split:<10} : {len(ds_raw[split])}")


def preprocess(batch):
    sources = [ex.get(SRC_LANG) or "" for ex in batch["translation"]]
    targets = [ex.get(TGT_LANG) or "" for ex in batch["translation"]]

    model_inputs = tokenizer(
        sources,
        text_target=targets,
        max_length=MAX_INPUT_LEN,
        truncation=True,
    )

    # Tronquer les labels à MAX_TARGET_LEN (max_target_length n'est pas supporté ici)
    model_inputs["labels"] = [
        ids[:MAX_TARGET_LEN] for ids in model_inputs["labels"]
    ]

    return model_inputs


tokenized = ds_raw.map(
    preprocess,
    batched=True,
    remove_columns=ds_raw["train"].column_names,
    desc="Tokenisation",
)
print(tokenized)

In [ ]:
tokenized["train"]["input_ids"]

In [ ]:
# DataCollatorForSeq2Seq : pad chaque batch à la longueur max DU BATCH
# (évite de pré-padder à une longueur fixe globale, moins de calcul inutile)
# pad_to_multiple_of=8 : optimise les Tensor Cores sur GPU Ampere+
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

## 6. Configuration LoRA

**LoRA** (Low-Rank Adaptation) gèle tous les poids originaux et ajoute deux petites matrices
$A \in \mathbb{R}^{r \times d_{in}}$ et $B \in \mathbb{R}^{d_{out} \times r}$ telles que :

$$\Delta W = BA, \quad \text{rang}(\Delta W) = r \ll d$$

Seuls $A$ et $B$ sont entraînés → environ **0.5 % des paramètres** du modèle total.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

peft_config = LoraConfig(
    task_type       = TaskType.SEQ_2_SEQ_LM,
    r               = LORA_R,                # rang : capacité d'adaptation
    lora_alpha      = LORA_ALPHA,             # facteur d'échelle de ΔW
    lora_dropout    = LORA_DROPOUT,           # régularisation
    target_modules  = LORA_TARGET_MODULES,    # projections linéaires à adapter
    bias            = "none",
    use_rslora      = True,   # rsLoRA : échelle par α/√r (plus stable pour r élevé)
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
# Attendu : trainable params ≈ 3.5 M / 614 M (≈ 0.57 %)

## 7. Entraînement

`Seq2SeqTrainer` gère la boucle d'entraînement, les checkpoints, la mixed precision et le suivi des métriques de traduction.

In [ ]:
def compute_metrics(eval_preds):
    """Calcule BLEU et chrF++ à chaque eval_step pendant l'entraînement."""
    pred_ids, label_ids = eval_preds

    # Remplacer les -100 (padding labels) par le token pad du tokenizer
    label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

    predictions = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    references  = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    bleu = sacrebleu_metric.compute(
        predictions=predictions, references=[[r] for r in references]
    )
    chrf = chrf_metric.compute(
        predictions=predictions, references=[[r] for r in references], word_order=2
    )

    return {
        "bleu":   round(bleu["score"], 2),
        "chrf++": round(chrf["score"], 2),
    }

In [ ]:
# Sous-ensemble de validation pour l'éval pendant l'entraînement.
# Évaluer sur les 6700 exemples complets à chaque eval = ~30 min/éval (génération
# autoregressive). On échantillonne 500 exemples → évals rapides, suivi fiable.
eval_subset = tokenized["validation"].select(
    range(min(500, len(tokenized["validation"])))
)

training_args = Seq2SeqTrainingArguments(
    output_dir = OUTPUT_DIR,

    # ── Durée ────────────────────────────────────────────────────────────────
    num_train_epochs = NUM_EPOCHS,
    eval_steps       = 1000,  # évaluer toutes les 1000 étapes (éval = coûteuse)
    save_steps       = 1000,  # sauvegarder un checkpoint toutes les 1000 étapes
    logging_steps    = 50,    # afficher les logs (loss) toutes les 50 étapes

    # ── Batch & mémoire ──────────────────────────────────────────────────────
    per_device_train_batch_size = BATCH_SIZE_TRAIN,
    per_device_eval_batch_size  = BATCH_SIZE_EVAL,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    # gradient_checkpointing : recalcule les activations au lieu de les stocker.
    # Économise ~60 % de VRAM au prix de ~20 % de ralentissement.
    gradient_checkpointing = True,

    # ── Learning rate & scheduler ────────────────────────────────────────────
    learning_rate     = LEARNING_RATE,
    warmup_ratio      = WARMUP_RATIO,
    # cosine : montée linéaire pendant le warmup, puis décroissance en cosinus.
    # Meilleur que constant ou linéaire pour le fine-tuning.
    lr_scheduler_type = "cosine",
    weight_decay      = WEIGHT_DECAY,

    # ── Précision mixte ──────────────────────────────────────────────────────
    # fp16 : calcul en 16 bits → 2× plus rapide et 2× moins de VRAM.
    # Sur GPU Ampere+ (RTX 30xx, A100), remplacer par bf16=True (plus stable).
    fp16 = (device == "cuda"),

    # ── Génération pendant l'évaluation ─────────────────────────────────────
    # predict_with_generate=True : utilise model.generate() pour les métriques
    # (obligatoire pour BLEU/chrF++ qui comparent des phrases entières).
    predict_with_generate  = True,
    generation_max_length  = MAX_TARGET_LEN,
    generation_num_beams   = 1,   # greedy pendant l'entraînement → éval ~4× plus rapide

    # ── Stratégie de sauvegarde ──────────────────────────────────────────────
    eval_strategy          = "steps",   # renommé depuis evaluation_strategy
    save_strategy          = "steps",
    load_best_model_at_end = True,          # recharge automatiquement le meilleur ckpt
    metric_for_best_model  = "chrf++",      # chrF++ plus robuste que BLEU pour l'éwé
    greater_is_better      = True,
    save_total_limit       = 2,             # conserver seulement 2 checkpoints

    # ── Logs ──────────────────────────────────────────────────────────────────
    # disable_tqdm=True : sur Kaggle en arrière-plan, les barres tqdm ne se
    # rafraîchissent pas ; on s'appuie sur les logs texte (logging_steps).
    disable_tqdm           = True,
    logging_first_step     = True,
    report_to              = "none",   # désactiver W&B/TensorBoard
)

trainer = Seq2SeqTrainer(
    model              = model,
    args               = training_args,
    train_dataset      = tokenized["train"],
    eval_dataset       = eval_subset,
    processing_class   = tokenizer,   # renommé depuis 'tokenizer' dans transformers ≥ 4.46
    data_collator      = data_collator,
    compute_metrics    = compute_metrics,
)

print("Trainer configuré. Démarrage de l'entraînement…")

In [ ]:

# Détection automatique du dernier checkpoint pour reprendre l'entraînement
last_ckpt = None
output_path = Path(OUTPUT_DIR)
if output_path.is_dir():
    ckpts = sorted(
        [d for d in output_path.iterdir()
         if d.is_dir() and d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[-1]),
    )
    if ckpts:
        last_ckpt = str(ckpts[-1])
        print(f"Reprise depuis le checkpoint : {last_ckpt}")
    else:
        print("Aucun checkpoint trouvé — entraînement from scratch.")
else:
    print("Aucun checkpoint trouvé — entraînement from scratch.")

train_result = trainer.train(resume_from_checkpoint=last_ckpt)

# Sauvegarder uniquement l'adaptateur LoRA (~30 Mo, pas le modèle entier ~1.2 Go)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"\nAdaptateur LoRA sauvegardé dans : {ADAPTER_DIR}")
print(f"Étapes   : {train_result.global_step}")
print(f"Loss train finale : {train_result.training_loss:.4f}")


## 8. Évaluation Finale sur le Test Set

On fusionne l'adaptateur LoRA avec le modèle de base (`merge_and_unload`) pour obtenir un modèle standard — plus rapide à l'inférence.

In [ ]:
print("Chargement du meilleur modèle fine-tuné…")
base_model_ft = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model_ft      = PeftModel.from_pretrained(base_model_ft, ADAPTER_DIR)
model_ft      = model_ft.merge_and_unload()   # fusionne LoRA → modèle standard
model_ft      = model_ft.to(device)
model_ft.eval()

tokenizer_ft = AutoTokenizer.from_pretrained(ADAPTER_DIR)

print("=== FINE-TUNÉ — test complet ===")
ft_test = compute_metrics_on_split(model_ft, tokenizer_ft, ds_raw["test"])
print(f"  BLEU   : {ft_test['bleu']}")
print(f"  chrF++ : {ft_test['chrf++']}")

## 9. Comparaison Baseline vs Fine-Tuné

In [ ]:
# Mise à jour du fichier de résultats
with open(RESULTS_FILE, "r", encoding="utf-8") as f:
    results = json.load(f)

results["fine_tune"]["test"] = ft_test

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# ── Tableau comparatif ────────────────────────────────────────────────────────
b = results["baseline"]["test"]
f = results["fine_tune"]["test"]

delta_bleu = round(f["bleu"]   - b["bleu"],   2)
delta_chrf = round(f["chrf++"] - b["chrf++"], 2)

print(f"{'Métrique':<12} {'Baseline':>10} {'Fine-tuné':>10} {'Δ':>8}")
print("-" * 44)
print(f"{'BLEU':<12} {b['bleu']:>10} {f['bleu']:>10} {delta_bleu:>+8}")
print(f"{'chrF++':<12} {b['chrf++']:>10} {f['chrf++']:>10} {delta_chrf:>+8}")
print(f"\nRésultats complets : {RESULTS_FILE}")

In [ ]:
# ── Quelques exemples de traduction ──────────────────────────────────────────
print("=== Exemples de traductions (test set) ===")
sample   = ds_raw["test"].select(range(5))
sources  = [ex["translation"][SRC_LANG] for ex in sample]
refs     = [ex["translation"][TGT_LANG] for ex in sample]
preds    = translate_batch(model_ft, tokenizer_ft, sources, SRC_LANG, TGT_LANG)

for src, ref, pred in zip(sources, refs, preds):
    print(f"\nSource    : {src}")
    print(f"Référence : {ref}")
    print(f"Prédit    : {pred}")